LoRA(Low-Rank Abstraction) it is used during fine tuning of LLMS especially during downstream tasks when we're talking about lower rank matrices(rank 1 but actually we go with rank 4/8) also I discovered after a vary long time where is SVD(Single Value Decomposition is used) , it was actually funny reading this paper as there was just few notations of maths like log of product of probabs and matrices quite obv isn't it?
anyways will write a learning on it.
:-)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class Linear(nn.Linear):
    """nn.Linear with a LoRA update: y = Wx + b + (alpha/r) * B(A(x))"""

    def __init__(self, in_features, out_features, r=0, lora_alpha=1,
                 lora_dropout=0.0, merge_weights=True, **kwargs):
        super().__init__(in_features, out_features, **kwargs)
        self.r = r
        self.merge_weights = merge_weights
        self.merged = False
        self.lora_dropout = nn.Dropout(lora_dropout) if lora_dropout > 0 else nn.Identity()

        if r > 0:
            self.lora_A = nn.Parameter(self.weight.new_zeros((r, in_features)))
            self.lora_B = nn.Parameter(self.weight.new_zeros((out_features, r)))
            self.scaling = lora_alpha / r
            self.weight.requires_grad = False          # freeze W0
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B)                # delta starts at zero

    def train(self, mode=True):
        super().train(mode)
        if self.r > 0 and self.merge_weights:
            delta = (self.lora_B @ self.lora_A) * self.scaling
            if mode and self.merged:                   # unmerge to train
                self.weight.data -= delta
                self.merged = False
            elif not mode and not self.merged:         # merge for fast inference
                self.weight.data += delta
                self.merged = True
        return self

    def forward(self, x):
        out = F.linear(x, self.weight, bias=self.bias)
        if self.r > 0 and not self.merged:
            out = out + (self.lora_dropout(x) @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return out


def mark_only_lora_as_trainable(model: nn.Module):
    for n, p in model.named_parameters():
        p.requires_grad = "lora_" in n


def apply_lora(model, r=8, alpha=16, dropout=0.0,
               targets=("q_proj", "v_proj")):
    """Swap target nn.Linear layers for LoRA Linear, copying pretrained weights."""
    for module in list(model.modules()):
        for name, child in list(module.named_children()):
            if name in targets and isinstance(child, nn.Linear) and not isinstance(child, Linear):
                new = Linear(child.in_features, child.out_features, r=r,
                             lora_alpha=alpha, lora_dropout=dropout,
                             bias=child.bias is not None)
                new = new.to(device=child.weight.device, dtype=child.weight.dtype)
                new.weight.data.copy_(child.weight.data)
                if child.bias is not None:
                    new.bias.data.copy_(child.bias.data)
                setattr(module, name, new)
    mark_only_lora_as_trainable(model)
    return model


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return train, total

Instruction Fine Tuning(Supervised Fine tuning) one advice don't run it if you don't have a gpu 30/50 or 40/50

In [ ]:

from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer



NAME = "HuggingFaceTB/SmolLM2-135M"
USE_LORA = False          
EPOCHS, BATCH, LR, MAX_LEN = 1, 4, 5e-5, 512
device = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(NAME).to(device)

if USE_LORA:
    apply_lora(model, r=8, alpha=16)
    LR = 2e-4             # LoRA tolerates a much higher LR
trainable, total = count_params(model)
print(f"trainable {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

N_SAMPLES = 2000          
ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split=f"train_sft[:{N_SAMPLES}]")
data = [(ex["prompt"] + "\n\nAnswer:", " " + ex["chosen"][-1]["content"]) for ex in ds]
data = [d for d in data if len(tok(d[0] + d[1]).input_ids) < MAX_LEN]
print(f"{len(data)} / {N_SAMPLES} samples kept (under {MAX_LEN} tokens)")


def collate(batch):
    ids, labels = [], []
    for prompt, resp in batch:
        p = tok(prompt, add_special_tokens=False).input_ids
        r = tok(resp, add_special_tokens=False).input_ids + [tok.eos_token_id]
        x = (p + r)[:MAX_LEN]
        y = ([-100] * len(p) + r)[:MAX_LEN]        
        ids.append(x); labels.append(y)
    L = max(map(len, ids))
    input_ids = torch.tensor([x + [tok.pad_token_id] * (L - len(x)) for x in ids])
    labels = torch.tensor([y + [-100] * (L - len(y)) for y in labels])
    attn = torch.tensor([[1] * len(x) + [0] * (L - len(x)) for x in ids])
    return {"input_ids": input_ids.to(device),
            "attention_mask": attn.to(device),
            "labels": labels.to(device)}


opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
model.train()
for epoch in range(EPOCHS):
    for batch in DataLoader(data, batch_size=BATCH, shuffle=True, collate_fn=collate):
        loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); opt.zero_grad()
        print(f"epoch {epoch} loss {loss.item():.3f}")

model.save_pretrained("sft_ckpt")
tok.save_pretrained("sft_ckpt")


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

trainable 134,515,008 / 134,515,008 (100.00%)
1260 / 2000 samples kept (under 512 tokens)
epoch 0 loss 1.600
epoch 0 loss 1.265
epoch 0 loss 1.940
epoch 0 loss 1.589
epoch 0 loss 2.278
epoch 0 loss 2.638
epoch 0 loss 0.997
epoch 0 loss 1.566
